# Stealth fix (C0 dormancy term) — seed 42

Pre-registration: `research/campaigns/runtime_h2o_validity/PREREG_stealth_fix_2026_11_23.md`.
Commit `8bb7c5f26b9f67ce295e32d8ec9fe34c5866c672`. Delayed onset (1 benign token then marker), r=0.5, seed 42.
**C0 term: c0_kl=0.0, c0_hinge=3.0** (passed to build so base logp is cached, and to train).
Go/no-go: PASS if C0 <= 0.05 AND runtime_h2o - C0 >= 0.20 with CI above 0.
Order: smoke -> build 300 -> 3 epochs -> eval 160 held-out dev prompts.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

REPO = "https://github.com/kartikbindra/btp-backdoor-research"
COMMIT = "8bb7c5f26b9f67ce295e32d8ec9fe34c5866c672"
WORK = Path("/kaggle/working")
SRC = Path("/kaggle/tmp/btp-research")
gpu = subprocess.check_output(["nvidia-smi","--query-gpu=name,uuid,driver_version,compute_cap","--format=csv,noheader"], text=True).strip()
print(gpu); print(sys.version); print(platform.platform())
assert "Tesla" in gpu or "NVIDIA" in gpu
SRC.parent.mkdir(parents=True, exist_ok=True)
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git","clone","--quiet",REPO,str(SRC)], check=True)
subprocess.run(["git","-C",str(SRC),"checkout","--quiet",COMMIT], check=True)
head = subprocess.check_output(["git","-C",str(SRC),"rev-parse","HEAD"], text=True).strip()
print("repo head", head); assert head == COMMIT, head
for rel in ("src/pfseb/runtime_train.py","scripts/run_runtime_train.py","src/pfseb/step3_c007.py"):
    assert (SRC / rel).exists(), f"missing {rel}"
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "cuda", torch.cuda.is_available())
(WORK/"session_env.json").write_text(json.dumps({"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__, "repo_head": head}, indent=2))


In [ ]:
def run_phase(tag, args, max_minutes=0):
    cmd = [sys.executable, "-u", "-m"] + args
    if max_minutes:
        cmd += ["--max_minutes", str(max_minutes)]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    with (WORK / f"{tag}.log").open("w", buffering=1) as log:
        proc = subprocess.run(cmd, cwd=str(SRC), stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit", proc.returncode, "minutes", round((time.time()-started)/60, 1), flush=True)
    return proc.returncode


In [ ]:
OUT = WORK / "rt_stealthfix_s42_v4"; OUT.mkdir(parents=True, exist_ok=True)
SMOKE = WORK / "smoke"; SMOKE.mkdir(exist_ok=True)
rc1 = run_phase("smoke_build", ["scripts.run_runtime_train","--phase","build","--out_dir",str(SMOKE),
    "--n_train","2","--bins","long_512","--retention","0.5","--benign_len","32","--onset_delay","1",
    "--c0_kl","0.0"])
rc2 = run_phase("smoke_train", ["scripts.run_runtime_train","--phase","train","--out_dir",str(SMOKE),
    "--epochs","1","--select_n","1","--select_tokens","4","--bins","long_512","--retention","0.5",
    "--benign_len","32","--onset_delay","1","--seed","42","--c0_kl","0.0","--c0_hinge","3.0"])
rc3 = run_phase("smoke_eval", ["scripts.run_runtime_train","--phase","eval","--out_dir",str(SMOKE),
    "--bins","long_512","--max_items","1","--max_new_tokens","8","--retention","0.5",
    "--benign_len","32","--onset_delay","1","--seed","42"])
PROCEED = rc1==0 and rc2==0 and rc3==0 and (SMOKE/"theta_b.pt").exists()
try:
    import json as _j
    h = _j.loads((SMOKE/"history.json").read_text())
    print("smoke c0_kl_mean", h[-1].get("c0_kl_mean"), "c0_hinge_mean", h[-1].get("c0_hinge_mean"))
except Exception as e:
    print("history read:", e)
print("smoke passed:", PROCEED)

In [ ]:
if PROCEED:
    run_phase("build", ["scripts.run_runtime_train","--phase","build","--out_dir",str(OUT),
        "--n_train","300","--bins","long_512,long_1024","--retention","0.5","--benign_len","32",
        "--onset_delay","1","--seed","42","--c0_kl","0.0"], max_minutes=45)
else:
    print("skipped: smoke failed")

In [ ]:
if PROCEED and (OUT/"examples.pt").exists():
    run_phase("train", ["scripts.run_runtime_train","--phase","train","--out_dir",str(OUT),
        "--epochs","3","--select_n","12","--select_tokens","8","--bins","long_512","--retention","0.5",
        "--benign_len","32","--onset_delay","1","--seed","42","--c0_kl","0.0","--c0_hinge","3.0"], max_minutes=220)
else:
    print("skipped: no examples")

In [ ]:
if (OUT/"theta_b.pt").exists():
    run_phase("eval", ["scripts.run_runtime_train","--phase","eval","--out_dir",str(OUT),"--pool","dev",
        "--bins","long_512","--skip_items","12","--max_items","160","--max_new_tokens","40",
        "--retention","0.5","--benign_len","32","--onset_delay","1","--seed","42"], max_minutes=150)
    print((OUT/"summary.json").read_text()[:3000])
else:
    print("no adapter")

In [ ]:
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
bundle = WORK / f"rt_stealthfix_s42_v4_{stamp}"; bundle.mkdir(exist_ok=True)
for root in [p for p in WORK.iterdir() if p.is_dir() and p.name != bundle.name]:
    if root.name in {"smoke", "rt_stealthfix_s42_v4"}:
        shutil.copytree(root, bundle / root.name, dirs_exist_ok=True)
for item in WORK.iterdir():
    if item.is_file() and (item.suffix in {".json",".log"} or item.name.endswith(".jsonl")):
        shutil.copy2(item, bundle / item.name)
shutil.make_archive(str(WORK / "rt_stealthfix_s42_v4_outputs"), "zip", bundle)
print("packaged", bundle, sorted(p.name for p in bundle.iterdir()))
